# NB2 — Feature Representations and Target Definition


This is an English, curated retrospective of the first bootcamp project. It follows the original presentation and retains the experiments needed to understand the selected model. It does not include the later Claude Code/GitHub upgrade.

**Two kinds of evidence:** snapshot statistics are computed from the supplied CSV; historical model metrics are transcribed from the original notebooks' stored outputs and explicitly labeled. The models have not been retrained for this edition. Set the relevant run flag to opt into an adapted historical rerun; scores may differ because the original `df_keyword.csv`, environment, and all intermediate states are unavailable. The supplied `CleanedDataset.csv` is used as a documented substitute, not asserted to be byte-identical to that intermediate file.

Cells carry original filenames and zero-based cell references in metadata. No API call or model download runs by default. The original evaluation limitations are noted without silently replacing the first project with an improved pipeline.


In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

# Place the supplied CSV files in data/, or set RASFF_DATA_DIR.
DATA_DIR = Path(os.environ.get("RASFF_DATA_DIR", "data"))
clean_path = DATA_DIR / "CleanedDataset.csv"
if not clean_path.exists():
    raise FileNotFoundError("Place the original CleanedDataset.csv in data/ or set RASFF_DATA_DIR.")
df = pd.read_csv(clean_path).reset_index(drop=True)
RUN_EXPERIMENTS = False
RUN_LLM_SAMPLE = False
SAVE_ARTIFACTS = False
print(f"Provided clean snapshot: {df.shape[0]:,} rows x {df.shape[1]} columns")


Provided clean snapshot: 27,397 rows x 23 columns


## 1. Define the classification task
The original six recorded labels were explored directly, then regrouped into three and two classes. The three-class grouping was an exploratory project choice; `undecided` is a decision status and the original labels are not a validated ordinal risk scale.

For the selected binary task, 1 means recorded as `serious`; 0 means one of the other five recorded labels. **Class 0 does not mean safe or hazard-free.**


In [2]:
THREE_CLASS = {"no risk": 0, "not serious": 0,
               "potential risk": 1, "undecided": 1, "potentially serious": 1,
               "serious": 2}
BINARY_CLASS = {label: int(label == "serious") for label in THREE_CLASS}
df["risk_3class"] = df["risk_decision"].map(THREE_CLASS)
df["risk_binary"] = df["risk_decision"].map(BINARY_CLASS)
assert df["risk_binary"].notna().all(), "Unexpected risk label: review before modeling."
display(pd.DataFrame({"Recorded label": list(THREE_CLASS),
                      "Exploratory 3-class label": list(THREE_CLASS.values()),
                      "Binary label": [BINARY_CLASS[label] for label in THREE_CLASS]}))
display(df["risk_binary"].value_counts().sort_index().to_frame("Notifications"))
print("Whole-snapshot majority baseline:", round(df["risk_binary"].value_counts(normalize=True).max(), 4))


Whole-snapshot majority baseline: 0.5386


,Recorded label,Exploratory 3-class label,Binary label
0,no risk,0,0
1,not serious,0,0
2,potential risk,1,0
3,undecided,1,0
4,potentially serious,1,0
5,serious,2,1


,Notifications
risk_binary,
0,12641
1,14756


## 2. Match numerical representations to the information
| Information | Early representation | Later selected representation |
|---|---|---|
| Subject sentence | Sentence embedding | all-MiniLM-L6-v2, 384 dimensions |
| Product category | Sentence embedding / one-hot | Rare grouping, then one-hot |
| Broad hazard type | One-hot | One-hot |
| Multiple origins | Multi-hot | Target encoding of the recorded origin string |
| Notifying country | Categorical encoding | Target encoding |
| Normalized hazard name | High-cardinality categorical field | Target encoding |
| Notification classification | One-hot | One-hot |

Multi-hot separates countries within a multi-origin string. The final model used a different representation, target encoding of that string; it did not keep the origin multi-hot block. This evolution matters more than presenting every encoding trial.


In [3]:
from sklearn.preprocessing import MultiLabelBinarizer
origin_lists = df["origin"].fillna("").str.split(r"[,/]+", regex=True).map(
    lambda items: [item.strip() for item in items if item.strip()])
origin_mlb = MultiLabelBinarizer()
origin_matrix = origin_mlb.fit_transform(origin_lists)
print("Origin multi-hot dimensions:", origin_matrix.shape)
display(pd.DataFrame(origin_matrix[:5], columns=origin_mlb.classes_).loc[:,
        origin_matrix[:5].any(axis=0)].astype(int))
display(pd.get_dummies(df["Hazard_Type"], prefix="hazard").head(3))


Origin multi-hot dimensions: (27397, 166)


,Belgium,Italy,Netherlands,United Kingdom,United States
0,0,0,1,0,0
1,1,0,0,0,0
2,0,0,0,0,1
3,0,1,0,0,0
4,0,0,0,1,0


,hazard_Allergen,hazard_Chemical/Contaminants,hazard_Fraud/Adulteration,hazard_Microbiological,hazard_Migration from Packaging,hazard_Mycotoxins,hazard_Novel/Unauthorised Ingredient,hazard_Others,hazard_Physical Contaminants
0,False,False,False,True,False,False,False,False,False
1,False,False,False,False,False,False,False,True,False
2,False,True,False,False,False,False,False,False,False


### Optional: inspect the chosen subject embedding
This small demonstration requires sentence-transformers and a model download. It shows the selected representation, without rerunning a model comparison. No embedding result is prefilled as if newly computed.


In [4]:
if RUN_EXPERIMENTS:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    sample_embeddings = embedder.encode(df["subject"].head(5).tolist(), convert_to_numpy=True)
    print("Five subject embeddings:", sample_embeddings.shape)
else:
    print("Optional historical experiment skipped.")


Optional historical experiment skipped.


## 3. Historical target comparison under more consistent conditions
The presentation's 6/3/2-class graph mixed algorithms and split ratios. Here the retained table uses the original Logistic Regression runs for all three targets, with the same six raw input fields and stratified random 70/30 split (seed 42). The outputs were rounded to four decimals in the original file.

Reducing the number of labels changes the task. Higher binary accuracy is not evidence that the binary task retains all distinctions or is automatically more useful for food safety.


In [5]:
recorded_target_results = pd.DataFrame([
    {"Target": "6 original labels", "Model": "Logistic Regression", "Split": "Random stratified 70/30", "Recorded accuracy": 0.6899, "Original cell": 10},
    {"Target": "3 exploratory groups", "Model": "Logistic Regression", "Split": "Random stratified 70/30", "Recorded accuracy": 0.7563, "Original cell": 8},
    {"Target": "Serious vs other labels", "Model": "Logistic Regression", "Split": "Random stratified 70/30", "Recorded accuracy": 0.8549, "Original cell": 4},
])
display(recorded_target_results)


,Target,Model,Split,Recorded accuracy,Original cell
0,6 original labels,Logistic Regression,Random stratified 70/30,0.6899,10
1,3 exploratory groups,Logistic Regression,Random stratified 70/30,0.7563,8
2,Serious vs other labels,Logistic Regression,Random stratified 70/30,0.8549,4


### Optional historical rerun: categorical baseline
The original baseline one-hot encoded `category`, `type`, `subject`, `origin`, `classification`, and `hazards`, producing a very wide matrix (27,418 columns in the stored output). Full subject strings treated as categories do not capture sentence similarity.

The adapted rerun below uses sparse one-hot columns to reduce memory use, but preserves the raw fields, target groupings, and split. Its results are new rerun results and must not replace the recorded table silently.


In [6]:
if RUN_EXPERIMENTS:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.linear_model import LogisticRegression
    from sklearn.multiclass import OneVsRestClassifier
    from sklearn.metrics import accuracy_score, f1_score
    FEATURES = ["category", "type", "subject", "origin", "classification", "hazards"]
    ohe = OneHotEncoder(drop="first", sparse_output=True)
    X = ohe.fit_transform(df[FEATURES].fillna("missing"))
    rerun_results = []
    for label, target in [("6 original labels", df["risk_decision"]),
                          ("3 exploratory groups", df["risk_3class"]),
                          ("Serious vs other labels", df["risk_binary"])]:
        X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.3, random_state=42, stratify=target)
        clf = LogisticRegression(solver="liblinear", random_state=42, max_iter=1000)
        if target.nunique() > 2:
            clf = OneVsRestClassifier(clf)
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        rerun_results.append({"Target": label, "Rerun accuracy": accuracy_score(y_test, pred),
                              "Rerun macro F1": f1_score(y_test, pred, average="macro")})
    display(pd.DataFrame(rerun_results))
else:
    print("Optional historical experiment skipped.")


Optional historical experiment skipped.


## 4. What this exploration contributed
The project moved toward a focused binary classifier for recorded serious decisions. The full snapshot contains 14,756 serious and 12,641 other records: a 53.86% / 46.14% split. The small classes in the original six-label task should not be described as extreme imbalance in the binary task.

The next notebook compares feature combinations and algorithms. Its early time split and later random split results are kept in separate tables.
